# AURORA — Skill Gap EDA & Course Catalog Analysis

This notebook performs exploratory analysis on:
- The course catalog (coverage across domains, level distribution)
- The skill taxonomy (most common skills across JDs)
- BKT parameter sensitivity analysis

Run this notebook to understand the data powering AURORA before building/optimizing the backend.

In [ ]:
import json
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as mplstyle
import seaborn as sns
import numpy as np
from collections import Counter

# Add backend to path
sys.path.insert(0, '../backend')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#1a1d26'
plt.rcParams['figure.facecolor'] = '#0d0f14'
plt.rcParams['text.color'] = '#f0f2f8'
plt.rcParams['axes.labelcolor'] = '#8892aa'
plt.rcParams['xtick.color'] = '#8892aa'
plt.rcParams['ytick.color'] = '#8892aa'
plt.rcParams['axes.edgecolor'] = '#242838'
plt.rcParams['grid.color'] = '#242838'
plt.rcParams['axes.grid'] = True

ACCENT = '#f0a500'
GREEN  = '#2dd4a0'
BLUE   = '#5b8ef0'
RED    = '#f05050'

print('✓ Imports OK')

## 1. Course Catalog Overview

In [ ]:
with open('../data/course_catalog.json') as f:
    catalog_raw = json.load(f)

catalog = pd.DataFrame(catalog_raw['courses'])
print(f'Total courses: {len(catalog)}')
print(f'Columns: {list(catalog.columns)}')
catalog[['id', 'name', 'category', 'level', 'duration_hours']].head(10)

In [ ]:
# Distribution by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_counts = catalog['category'].value_counts()
axes[0].barh(cat_counts.index, cat_counts.values, color=ACCENT)
axes[0].set_title('Courses by Category', color='#f0f2f8', pad=12)
axes[0].set_xlabel('Count')

level_counts = catalog['level'].value_counts()
colors = [GREEN, ACCENT, RED]
axes[1].pie(level_counts.values, labels=level_counts.index, autopct='%1.0f%%',
            colors=colors[:len(level_counts)], textprops={'color': '#f0f2f8'})
axes[1].set_title('Courses by Difficulty Level', color='#f0f2f8', pad=12)

plt.tight_layout()
plt.savefig('../docs/catalog_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Hours distribution
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(catalog['duration_hours'], bins=15, color=BLUE, edgecolor='#0d0f14', alpha=0.9)
ax.set_title('Distribution of Course Duration (hours)', color='#f0f2f8', pad=12)
ax.set_xlabel('Hours')
ax.set_ylabel('Number of Courses')
ax.axvline(catalog['duration_hours'].mean(), color=ACCENT, linestyle='--',
           label=f"Mean: {catalog['duration_hours'].mean():.1f}h")
ax.legend()
plt.tight_layout()
plt.show()
print(catalog['duration_hours'].describe())

## 2. Skill Coverage Analysis

In [ ]:
# Flatten skills_taught across all courses
all_skills_taught = []
for _, row in catalog.iterrows():
    all_skills_taught.extend(row['skills_taught'])

skill_freq = Counter(all_skills_taught)
top_skills = pd.DataFrame(skill_freq.most_common(20), columns=['skill', 'course_count'])

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_skills['skill'][::-1], top_skills['course_count'][::-1], color=ACCENT)
ax.set_title('Top 20 Most Covered Skills in Catalog', color='#f0f2f8', pad=12)
ax.set_xlabel('Number of Courses Covering This Skill')
plt.tight_layout()
plt.show()

In [ ]:
# Prerequisite graph depth analysis
import networkx as nx

G = nx.DiGraph()
for _, row in catalog.iterrows():
    G.add_node(row['id'])
    for prereq in row['prerequisites']:
        G.add_edge(prereq, row['id'])

print(f'Nodes (courses): {G.number_of_nodes()}')
print(f'Edges (prereq links): {G.number_of_edges()}')
print(f'Has cycles: {not nx.is_directed_acyclic_graph(G)} (should be False!)')

# Compute depth of each course in the graph
depths = {}
for node in nx.topological_sort(G):
    preds = list(G.predecessors(node))
    depths[node] = 0 if not preds else max(depths.get(p, 0) for p in preds) + 1

depth_series = pd.Series(depths)
print(f'\nDepth distribution:\n{depth_series.value_counts().sort_index()}')

## 3. BKT Parameter Sensitivity Analysis

In [ ]:
from knowledge_tracing import update_knowledge_state, initialize_from_proficiency

def simulate_learning(p_learn, n_correct=20, initial_level=1):
    state = initialize_from_proficiency('X', initial_level)
    state.p_learn = p_learn
    trajectory = [state.p_mastery]
    for _ in range(n_correct):
        state = update_knowledge_state(state, correct=True)
        trajectory.append(state.p_mastery)
    return trajectory

p_learn_values = [0.1, 0.2, 0.3, 0.4, 0.5]
fig, ax = plt.subplots(figsize=(12, 5))

colors_grad = plt.cm.plasma(np.linspace(0.2, 0.9, len(p_learn_values)))
for p_learn, color in zip(p_learn_values, colors_grad):
    traj = simulate_learning(p_learn)
    ax.plot(traj, label=f'P_learn={p_learn}', color=color, linewidth=2)

ax.axhline(0.95, color=RED, linestyle='--', alpha=0.7, label='Mastery threshold (0.95)')
ax.set_title('BKT: Mastery Trajectory vs P(learn) Parameter', color='#f0f2f8', pad=12)
ax.set_xlabel('Practice Opportunities')
ax.set_ylabel('P(Mastery)')
ax.legend(facecolor='#1a1d26', edgecolor='#242838', labelcolor='#f0f2f8')
plt.tight_layout()
plt.savefig('../docs/bkt_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Simulated Gap Analysis on Sample Resumes

In [ ]:
from gap_analyzer import compute_skill_gap
from pathway_engine import generate_learning_pathway, load_course_catalog
from metrics import evaluate_pathway

catalog = load_course_catalog()

# Simulate 3 candidate profiles
profiles = [
    {
        'name': 'Junior DS',
        'resume': {'skills': [
            {'skill_name': 'Python', 'proficiency_level': 2},
            {'skill_name': 'Statistics', 'proficiency_level': 2},
        ]}
    },
    {
        'name': 'Mid DS',
        'resume': {'skills': [
            {'skill_name': 'Python', 'proficiency_level': 4},
            {'skill_name': 'Machine Learning', 'proficiency_level': 3},
            {'skill_name': 'SQL', 'proficiency_level': 3},
            {'skill_name': 'Pandas', 'proficiency_level': 3},
        ]}
    },
    {
        'name': 'Senior ML Eng',
        'resume': {'skills': [
            {'skill_name': 'Python', 'proficiency_level': 5},
            {'skill_name': 'Machine Learning', 'proficiency_level': 4},
            {'skill_name': 'PyTorch', 'proficiency_level': 4},
            {'skill_name': 'SQL', 'proficiency_level': 4},
            {'skill_name': 'Docker', 'proficiency_level': 3},
            {'skill_name': 'MLOps', 'proficiency_level': 3},
        ]}
    },
]

target_jd = {
    'job_title': 'Senior ML Engineer',
    'seniority': 'senior',
    'skills': [
        {'skill_name': 'Python',          'required_level': 4, 'is_mandatory': True,  'category': 'programming', 'context': ''},
        {'skill_name': 'Machine Learning', 'required_level': 4, 'is_mandatory': True,  'category': 'data-science', 'context': ''},
        {'skill_name': 'PyTorch',          'required_level': 4, 'is_mandatory': True,  'category': 'data-science', 'context': ''},
        {'skill_name': 'SQL',              'required_level': 3, 'is_mandatory': True,  'category': 'data-engineering', 'context': ''},
        {'skill_name': 'Docker',           'required_level': 3, 'is_mandatory': True,  'category': 'cloud-devops', 'context': ''},
        {'skill_name': 'MLOps',            'required_level': 3, 'is_mandatory': False, 'category': 'data-science', 'context': ''},
    ]
}

results = []
for profile in profiles:
    gap = compute_skill_gap(profile['resume'], target_jd)
    pathway = generate_learning_pathway(gap, target_jd, catalog)
    eval_result = evaluate_pathway(gap, pathway, catalog)
    results.append({
        'Profile':          profile['name'],
        'Gaps':             gap['gap_summary']['total_gaps'],
        'Coverage%':        gap['gap_summary']['coverage_pct'],
        'Total Hours':      pathway['estimated_total_hours'],
        'Weeks':            pathway['estimated_weeks'],
        'Composite Score':  eval_result['composite_score'],
        'Hours Saved':      eval_result['training_savings']['hours_saved'],
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
# Bar chart: hours per profile
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df['Profile'], df['Total Hours'], color=[GREEN, ACCENT, BLUE])
axes[0].set_title('Personalized Training Hours by Profile', color='#f0f2f8', pad=12)
axes[0].set_ylabel('Hours')

axes[1].bar(df['Profile'], df['Composite Score'], color=[GREEN, ACCENT, BLUE])
axes[1].set_title('Pathway Quality (Composite Score / 100)', color='#f0f2f8', pad=12)
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('../docs/profile_comparison.png', dpi=150, bbox_inches='tight')
plt.show()